# Option 1: Fine-tune InLegalBERT for SCOTUS Vote Prediction

Fine-tunes `law-ai/InLegalBERT` on the extracted SCOTUS dataset.  
- **Split**: temporal — train < 2010, val 2010–2014, test >= 2015  
- **Output**: `checkpoints/option1/best_model/` and `checkpoints/option1/test_predictions.csv`

In [ ]:
!pip install -q "accelerate>=0.26.0"

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

DATA_PATH = "data/extracted.csv"
OUTPUT_DIR = "checkpoints/option1"

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH=512
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-5
PATIENCE = 3
SEED =12345

## 2. Imports

In [ ]:
%pip install numpy pandas torch 
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback,)
from sklearn.metrics import (accuracy_score, f1_score, classification_report, precision_recall_fscore_support)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 3. Loading Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df[df["label"].isin([0, 1])].copy()
df["all_utterances"] = df["all_utterances"].fillna("")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year", "label"]).copy()
df["year"] = df["year"].astype(int)

print(f"Loaded {len(df):,} valid rows. Year range: {df['year'].min()}–{df['year'].max()}")
df.head(3)

## 4. Time Splitting to avoid data leakage

In [ ]:
train_df = df[df["year"] < 2010].reset_index(drop=True)
val_df = df[(df["year"] >= 2010) & (df["year"] < 2015)].reset_index(drop=True)
test_df =  df[df["year"] >= 2015].reset_index(drop=True)

for name, split in [("Train (<2010)", train_df), ("Val (2010–14)", val_df), ("Test (>=2015)", test_df)]:
    vc = split["label"].value_counts().sort_index()
    n  = len(split)
    print(f"{name:<16}: {n:6,} rows | label=0: {vc.get(0,0):,} ({vc.get(0,0)/n*100:.1f}%) | label=1: {vc.get(1,0):,} ({vc.get(1,0)/n*100:.1f}%)")

## 5. Model loading

In [ ]:
print(f"Loading tokenizer and model")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

print("Done")

## 6. Dataset tokenization

In [ ]:
class SCOTUSDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        texts = df["all_utterances"].fillna("").tolist()
        self.labels = df["label"].tolist()
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Tokenizing:")

train_dataset = SCOTUSDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SCOTUSDataset(val_df,   tokenizer, MAX_LENGTH)
test_dataset = SCOTUSDataset(test_df,  tokenizer, MAX_LENGTH)
print("Done")

## 7. Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "accuracy": acc,
        "f1": macro_f1,
        "f1_class0": float(per_class_f1[0]) if len(per_class_f1) > 0 else 0.0,
        "f1_class1": float(per_class_f1[1]) if len(per_class_f1) > 1 else 0.0,
    }

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=50,
    fp16=True,
    bf16=use_bf16,
    seed=12345,
    report_to="none",
    dataloader_num_workers=0,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("Starting training")
trainer.train()

## 8. Test set eval

In [ ]:
test_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_df["label"].tolist()
probs = torch.softmax(torch.tensor(test_output.predictions, dtype=torch.float32), dim=-1).numpy()

print(classification_report(test_labels, test_preds, target_names=["Respondent (0)", "Petitioner (1)"], digits=4,))

macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
acc = accuracy_score(test_labels, test_preds)

print(f"Overall:  accuracy: {acc:.4f},    macro-F1: {macro_f1:.4f}")

## 9. Storing predictions and best model

In [ ]:
pred_df = test_df[["case_id", "justice_id", "year", "label"]].copy()
pred_df["pred_label"] = test_preds
pred_df["prob_class0"] = probs[:, 0]
pred_df["prob_class1"] = probs[:, 1]
pred_df["correct"] = (pred_df["pred_label"] == pred_df["label"]).astype(int)

pred_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
pred_df.to_csv(pred_path, index=False)

print(f"Test predictions saved at: {pred_path}")

best_model_dir = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

print(f"Best model saved at: {best_model_dir}")